In [1]:
import numpy as np
import cv2 as cv
import open3d as o3d
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import urllib.request
import os

In [2]:
# Função para carregar imagens de URLs
def load_image_from_url(url):
    try:
        with urllib.request.urlopen(url) as response:
            img_array = np.asarray(bytearray(response.read()), dtype=np.uint8)
            return cv.imdecode(img_array, cv.IMREAD_COLOR)
    except Exception as e:
        print(f"Erro no URL {url}: {e}")
        return None

# URLs para as imagens KITTI
base_url = "https://raw.githubusercontent.com/Delevati/vc-2024-2/main/img/task04/q2"

# ID das cenas KITTI que vamos processar
kitti_scenes = ["000010", "000032", "000188"]

# Mapear URLs para as imagens e arquivos de calibração
image_urls = {}
for scene_id in kitti_scenes:
    image_urls[scene_id] = {
        'left': f"{base_url}/{scene_id}_10.png",
        'right': f"{base_url}/{scene_id}_11.png",
        'calib': f"{base_url}/{scene_id}.txt"
    }

# Função para carregar imagem de URL ou arquivo local
def load_image(path_or_url, use_url=True):
    if use_url:
        return load_image_from_url(path_or_url)
    else:
        return cv.imread(path_or_url)

# Função para ler arquivo de calibração KITTI
def read_kitti_calib_file(calib_path, use_url=True):
    if use_url:
        try:
            with urllib.request.urlopen(calib_path) as response:
                calib_content = response.read().decode('utf-8')
                lines = calib_content.strip().split('\n')
        except Exception as e:
            print(f"Erro ao ler arquivo de calibração {calib_path}: {e}")
            return None
    else:
        try:
            with open(calib_path, 'r') as f:
                lines = f.readlines()
        except Exception as e:
            print(f"Erro ao ler arquivo de calibração {calib_path}: {e}")
            return None
    
    # Processamento do arquivo de calibração
    calib_data = {}
    for line in lines:
        if line.strip() == '' or line.startswith('//'):
            continue
            
        key, value = line.split(':', 1)
        key = key.strip()
        values = [float(x) for x in value.strip().split()]
        
        # As matrizes de projeção são 3x4
        if key in ['P_rect_00', 'P_rect_01', 'P_rect_02', 'P_rect_03']:
            values = np.array(values).reshape(3, 4)
        elif key in ['R_rect_00', 'R_rect_01', 'R_rect_02', 'R_rect_03']:
            values = np.array(values).reshape(3, 3)
            
        calib_data[key] = values
    
    # Converte para o formato que usamos no código
    # P2 = câmera esquerda, P3 = câmera direita
    return {
        'P2': calib_data['P_rect_02'],
        'P3': calib_data['P_rect_03']
    }